
## Folder 04 - File 01: Silver Customers:


**Transformations:**

- Read bronze_customers.
- Trim spaces from text columns.
- Convert email addresses to lowercase.
- Convert created_at to a date.
- Remove duplicate customer_id records.
- Quarantine records with invalid or missing customer IDs or dates.
- Write valid records to silver_customers.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"

BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"

BRONZE_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.bronze_customers"
)

SILVER_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_customers"
)

QUARANTINE_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.quarantine_customers"
)


## 3. Create Silver schema

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"`{CATALOG}`.`{SILVER_SCHEMA}`"
)

print("Silver schema is ready.")


## 4. Read Bronze customers

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise Exception(
        f"Bronze table not found: {BRONZE_TABLE}. "
        "Run Folder 03 first."
    )

bronze_df = spark.table(BRONZE_TABLE)

print(f"Bronze customer records: {bronze_df.count()}")

display(bronze_df.limit(10))


## 5. Clean and standardize customer columns

cleaned_df = (
    bronze_df

    # Remove leading and trailing spaces

    .withColumn(
        "customer_id",
        F.trim(F.col("customer_id").cast("string"))
    )
    .withColumn(
        "first_name",
        F.trim(F.col("first_name").cast("string"))
    )
    .withColumn(
        "last_name",
        F.trim(F.col("last_name").cast("string"))
    )
    .withColumn(
        "email",
        F.lower(F.trim(F.col("email").cast("string")))
    )
    .withColumn(
        "phone",
        F.trim(F.col("phone").cast("string"))
    )
    .withColumn(
        "city",
        F.trim(F.col("city").cast("string"))
    )
    .withColumn(
        "state",
        F.trim(F.col("state").cast("string"))
    )

    # Convert created_at into DATE


    .withColumn(
        "created_at",
        F.to_date(F.col("created_at").cast("string"))
    )
)

print("Customer standardization completed.")

display(cleaned_df.limit(10))


## 6. Identify invalid customer records

invalid_condition = (
    F.col("customer_id").isNull()
    | (F.col("customer_id") == "")
    | F.col("created_at").isNull()
)

invalid_df = (
    cleaned_df
    .filter(invalid_condition)
    .withColumn(
        "_quarantine_reason",
        F.when(
            F.col("customer_id").isNull()
            | (F.col("customer_id") == ""),
            F.lit("Missing or empty customer_id")
        )
        .when(
            F.col("created_at").isNull(),
            F.lit("Missing or invalid created_at")
        )
    )
)

valid_df = cleaned_df.filter(~invalid_condition)

print(f"Valid records: {valid_df.count()}")
print(f"Invalid records: {invalid_df.count()}")


## 7. Remove duplicate customer records

deduplication_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("_ingestion_timestamp").desc_nulls_last(),
        F.col("_source_file").desc_nulls_last()
    )
)

deduplicated_df = (
    valid_df
    .withColumn(
        "_row_number",
        F.row_number().over(deduplication_window)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

print(
    f"Records after deduplication: "
    f"{deduplicated_df.count()}"
)


## 8. Save invalid records to quarantine

(
    invalid_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print(f"Quarantine table created: {QUARANTINE_TABLE}")


## 9. Save clean records to Silver Delta table

(
    deduplicated_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Silver table created: {SILVER_TABLE}")


## 10. Validate Silver customer records

silver_df = spark.table(SILVER_TABLE)

total_silver_records = silver_df.count()

duplicate_customer_groups = (
    silver_df
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

missing_customer_ids = (
    silver_df
    .filter(
        F.col("customer_id").isNull()
        | (F.col("customer_id") == "")
    )
    .count()
)

missing_created_dates = (
    silver_df
    .filter(F.col("created_at").isNull())
    .count()
)

print("=" * 50)
print("SILVER CUSTOMER VALIDATION")
print("=" * 50)

print(f"Silver records: {total_silver_records}")
print(f"Duplicate customer groups: {duplicate_customer_groups}")
print(f"Missing customer IDs: {missing_customer_ids}")
print(f"Missing created dates: {missing_created_dates}")

if (
    duplicate_customer_groups > 0
    or missing_customer_ids > 0
    or missing_created_dates > 0
):
    raise Exception("Silver customer validation failed.")

print("Silver customer validation passed.")


## 11. Display final Silver customer data

display(
    silver_df.orderBy("customer_id").limit(20)
)


 ## 12. Final summary

print("=" * 50)
print("CUSTOMER SILVER TRANSFORMATION COMPLETED")
print("=" * 50)

print(f"Bronze table     : {BRONZE_TABLE}")
print(f"Silver table     : {SILVER_TABLE}")
print(f"Quarantine table : {QUARANTINE_TABLE}")

print(f"Bronze records   : {bronze_df.count()}")
print(f"Silver records   : {total_silver_records}")
print(f"Invalid records  : {invalid_df.count()}")

print("=" * 50)
